In [7]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [8]:
!pip install -q transformers accelerate bitsandbytes

In [ ]:
# import os
# import gc
# import numpy as np
# import pandas as pd
# import torch
# from datasets import Dataset
# from dataclasses import dataclass
# from sklearn.model_selection import train_test_split
# from transformers import AutoTokenizer, AutoModelForMultipleChoice, Trainer, TrainingArguments
# from transformers.tokenization_utils_base import PreTrainedTokenizerBase, PaddingStrategy
# from typing import Optional, Union
# from peft import LoraConfig, get_peft_model, TaskType

# # ==============================================================================
# # 1. CONFIGURATION & DIRECTORIES
# # ==============================================================================
# MODEL_NAME = "microsoft/deberta-v3-large"
# MAX_INPUT_LENGTH = 256  # Optimized for speed and safety against OOM
# BATCH_SIZE = 2          # Batch size per device
# GRADIENT_ACCUMULATION_STEPS = 4  # Effective batch size of 8
# LEARNING_RATE = 2e-5
# NUM_EPOCHS = 2          # Fast, effective fine-tuning profile

# # Standard Kaggle relative input directory paths
# TRAIN_CSV_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
# TEST_CSV_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
# SUBMISSION_OUTPUT = "submission.csv"

# OPTION_TO_INDEX = {letter: idx for idx, letter in enumerate("ABCDE")}
# INDEX_TO_OPTION = {idx: letter for idx, letter in enumerate("ABCDE")}

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Using device: {device}")

# # ==============================================================================
# # 2. DATA PROCESSING & TOKENIZATION
# # ==============================================================================
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# def preprocess_mcq(examples):
#     first_sentences = []
#     second_sentences = []
#     labels = []
    
#     for i in range(len(examples['prompt'])):
#         prompt = examples['prompt'][i]
        
#         # Duplicate prompt 5 times for the 5 choices
#         first_sentences.append([prompt] * 5)
#         second_sentences.append([
#             str(examples['A'][i]),
#             str(examples['B'][i]),
#             str(examples['C'][i]),
#             str(examples['D'][i]),
#             str(examples['E'][i])
#         ])
        
#         if 'answer' in examples and examples['answer'][i] in OPTION_TO_INDEX:
#             labels.append(OPTION_TO_INDEX[examples['answer'][i]])
            
#     first_sentences = sum(first_sentences, [])
#     second_sentences = sum(second_sentences, [])
    
#     tokenized = tokenizer(
#         first_sentences,
#         second_sentences,
#         truncation=True,
#         max_length=MAX_INPUT_LENGTH,
#     )
    
#     reshaped = {k: [v[i:i+5] for i in range(0, len(v), 5)] for k, v in tokenized.items()}
#     if labels:
#         reshaped['label'] = labels
        
#     return reshaped

# @dataclass
# class DataCollatorForMultipleChoice:
#     tokenizer: PreTrainedTokenizerBase
#     padding: Union[bool, str, PaddingStrategy] = True
#     max_length: Optional[int] = None
#     pad_to_multiple_of: Optional[int] = None

#     def __call__(self, features):
#         label_name = "label" if "label" in features[0].keys() else "labels"
#         labels = [feature.pop(label_name) for feature in features] if label_name in features[0].keys() else None
#         batch_size = len(features)
#         num_choices = len(features[0]["input_ids"])
        
#         flattened_features = [
#             [{k: v[i] for k, v in feature.items()} for i in range(num_choices)] for feature in features
#         ]
#         flattened_features = sum(flattened_features, [])
        
#         batch = self.tokenizer.pad(
#             flattened_features,
#             padding=self.padding,
#             max_length=self.max_length,
#             pad_to_multiple_of=self.pad_to_multiple_of,
#             return_tensors="pt",
#         )
        
#         batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
#         if labels is not None:
#             batch["labels"] = torch.tensor(labels, dtype=torch.long)
#         return batch

# # MAP@3 Metric Function for validation
# def compute_metrics(eval_predictions):
#     predictions, label_ids = eval_predictions
#     sorted_preds = np.argsort(-predictions, axis=1)
    
#     map3_score = 0.0
#     for i in range(len(label_ids)):
#         true_idx = label_ids[i]
#         top_3_preds = sorted_preds[i][:3]
        
#         if true_idx == top_3_preds[0]:
#             map3_score += 1.0
#         elif true_idx == top_3_preds[1]:
#             map3_score += 0.5
#         elif true_idx == top_3_preds[2]:
#             map3_score += 0.333
            
#     return {"map@3": map3_score / len(label_ids)}

# # ==============================================================================
# # 3. CORE PROCESSING PIPELINE
# # ==============================================================================
# def run_pipeline():
#     # --- STEP 1: Load and Split Train Data ---
#     print("Loading training data...")
#     df_all = pd.read_csv(TRAIN_CSV_PATH)
    
#     # Stratified split to preserve validation control locally
#     df_train, df_val = train_test_split(df_all, test_size=0.15, random_state=42)
    
#     # FIXED: Added .tolist() to prevent the array truth value error
#     train_ds = Dataset.from_pandas(df_train).map(preprocess_mcq, batched=True, remove_columns=df_train.columns.tolist())
#     val_ds = Dataset.from_pandas(df_val).map(preprocess_mcq, batched=True, remove_columns=df_val.columns.tolist())
    
#     # --- STEP 2: Initialize Model with LoRA Weights ---
#     print("Initializing Model and applying LoRA layers...")
#     model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)
    
#     peft_config = LoraConfig(
#         r=16,
#         lora_alpha=32,
#         target_modules=["query_proj", "value_proj", "key_proj", "output_proj"],
#         lora_dropout=0.05,
#         bias="none",
#         task_type=TaskType.SEQ_CLS
#     )
#     model = get_peft_model(model, peft_config)
    
#     training_args = TrainingArguments(
#         output_dir="./results",
#         eval_strategy="epoch",            # <--- FIXED HERE
#         save_strategy="epoch",
#         learning_rate=LEARNING_RATE,
#         per_device_train_batch_size=BATCH_SIZE,
#         per_device_eval_batch_size=BATCH_SIZE * 2,
#         gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
#         num_train_epochs=NUM_EPOCHS,
#         weight_decay=0.01,
#         fp16=True,
#         logging_steps=20,
#         load_best_model_at_end=True,
#         metric_for_best_model="map@3",
#         greater_is_better=True,
#         report_to="none"
#     )
#     trainer = Trainer(
#         model=model,
#         args=training_args,
#         train_dataset=train_ds,
#         eval_dataset=val_ds,
#         processing_class=tokenizer,   # <--- FIXED HERE
#         data_collator=DataCollatorForMultipleChoice(tokenizer=tokenizer),
#         compute_metrics=compute_metrics,
#     ) 
    
#     # Execute fine-tuning on train.csv
#     print("Starting training process...")
#     trainer.train()
    
#     # --- STEP 3: Load Test Dataset and Perform Inference ---
#     print("Loading test data for ranking predictions...")
#     test_df = pd.read_csv(TEST_CSV_PATH)
    
#     # Prepare test set for model ingestion
#     test_ds = Dataset.from_pandas(test_df).map(
#         preprocess_mcq, 
#         batched=True, 
#         remove_columns=[col for col in test_df.columns if col != 'id']
#     )
    
#     model.eval()
#     test_collator = DataCollatorForMultipleChoice(tokenizer=tokenizer)
#     all_logits = []
    
#     print("Executing batched inference on test set...")
#     with torch.no_grad():
#         for i in range(0, len(test_ds), BATCH_SIZE * 4):
#             batch_features = [test_ds[j] for j in range(i, min(i + BATCH_SIZE * 4, len(test_ds)))]
#             batch = test_collator(batch_features)
            
#             inputs = {k: v.to(device) for k, v in batch.items()}
#             outputs = model(**inputs)
#             all_logits.append(outputs.logits.cpu().numpy())
            
#     all_logits = np.concatenate(all_logits, axis=0)
    
#     # --- STEP 4: Format Rank Predictions and Output Submission ---
#     print("Formatting submission file...")
#     sorted_indices = np.argsort(-all_logits, axis=1)[:, :3]
    
#     predictions = []
#     for row in sorted_indices:
#         top_3_letters = [INDEX_TO_OPTION[idx] for idx in row]
#         predictions.append(" ".join(top_3_letters))
        
#     submission_df = pd.DataFrame({
#         'id': test_df['id'],
#         'Prediction': predictions
#     })
    
#     submission_df.to_csv(SUBMISSION_OUTPUT, index=False)
#     print(f"Success! Final submission file written to {SUBMISSION_OUTPUT}")
#     print(submission_df.head())
# if __name__ == "__main__":
#     run_pipeline()

Using device: cpu
Loading training data...


Map:   0%|          | 0/1700 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Initializing Model and applying LoRA layers...


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight               

Starting training process...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [2]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# ── Load Data ──────────────────────────────────────────────────
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df  = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
OPTION_COLS = ["A", "B", "C", "D", "E"]

def mapk(actual, predicted, k=3):
    def apk(a, p):
        score, hits = 0.0, 0
        for i, pi in enumerate(p[:k]):
            if pi == a:
                hits += 1
                score += hits / (i + 1)
        return score
    return np.mean([apk(a, p) for a, p in zip(actual, predicted)])

# ── Build pairwise ranking features ───────────────────────────
# Key insight: instead of "is this option correct?" (binary)
# we ask "is option X better than option Y?" (ranking)
# This is what your friend did — rank-based not classify-based

def build_texts(df):
    """
    For each question, create 5 texts:
    prompt + option (word ngrams catch semantics,
    char ngrams catch subtle spelling/word differences)
    """
    texts = []
    for _, row in df.iterrows():
        for opt in OPTION_COLS:
            # Full text combination
            combined = (
                str(row["prompt"]) + " " +
                str(row["prompt"]) +  # repeat prompt to give it more weight
                " [SEP] " +
                str(row[opt])
            )
            texts.append(combined)
    return texts

train_texts = build_texts(train_df)
test_texts  = build_texts(test_df)

# ── Labels: rank-based (not binary 0/1) ───────────────────────
# Correct option gets score 4, others get 0,1,2,3 randomly
# This teaches the model to RANK not just classify
label2idx = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}

train_labels = []
for _, row in train_df.iterrows():
    correct_idx = label2idx[row["answer"]]
    for i in range(5):
        # Correct = 4 (highest rank), wrong = 0
        train_labels.append(4 if i == correct_idx else 0)

# ── TF-IDF: BOTH word and character ngrams ─────────────────────
# Word ngrams: catch semantic meaning
# Char ngrams: catch subtle differences in similar paragraphs
# This is the KEY difference from basic TF-IDF

from sklearn.pipeline import Pipeline, FeatureUnion

word_tfidf = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 3),      # unigrams, bigrams, trigrams
    max_features=200000,
    sublinear_tf=True,
    strip_accents="unicode",
    min_df=1
)

char_tfidf = TfidfVectorizer(
    analyzer="char_wb",       # character ngrams within word boundaries
    ngram_range=(3, 5),       # 3,4,5 char ngrams
    max_features=200000,
    sublinear_tf=True,
    strip_accents="unicode",
    min_df=1
)

combined_features = FeatureUnion([
    ("word", word_tfidf),
    ("char", char_tfidf)
])

# ── Train/Val Split ────────────────────────────────────────────
n = len(train_df)
idx = np.arange(n)
tr_idx, val_idx = train_test_split(idx, test_size=0.2, random_state=42)

tr_text_idx  = [i*5+j for i in tr_idx  for j in range(5)]
val_text_idx = [i*5+j for i in val_idx for j in range(5)]

tr_texts_split   = [train_texts[i] for i in tr_text_idx]
val_texts_split  = [train_texts[i] for i in val_text_idx]
tr_labels_split  = [train_labels[i] for i in tr_text_idx]

print("Fitting TF-IDF features (word + char ngrams)...")
X_train = combined_features.fit_transform(tr_texts_split)
X_val   = combined_features.transform(val_texts_split)
X_test  = combined_features.transform(test_texts)
print(f"Feature matrix shape: {X_train.shape}")

# ── Logistic Regression ────────────────────────────────────────
print("Training Logistic Regression...")
clf = LogisticRegression(
    C=5.0,
    max_iter=2000,
    solver="saga",
    n_jobs=-1
)
clf.fit(X_train, tr_labels_split)
print("Done!")

# ── Rank-based Prediction ──────────────────────────────────────
# Use predict_proba score for class 4 (correct option)
# This gives a continuous score per option → rank them

def get_scores_and_predict(X, n_questions):
    # Get probability of being the correct answer (class 4)
    probs = clf.predict_proba(X)
    # Find which column corresponds to class 4
    class4_idx = list(clf.classes_).index(4)
    scores = probs[:, class4_idx]
    # Reshape to (n_questions, 5)
    scores_matrix = scores.reshape(n_questions, 5)
    # Rank: top 3 per question
    top3 = np.argsort(-scores_matrix, axis=1)[:, :3]
    return [[OPTION_COLS[i] for i in row] for row in top3]

# ── Validate ───────────────────────────────────────────────────
val_preds  = get_scores_and_predict(X_val, len(val_idx))
true_labels = [train_df.iloc[i]["answer"] for i in val_idx]
val_map3   = mapk(true_labels, val_preds)
print(f"\nValidation MAP@3: {val_map3:.4f}")

# ── Test Submission ────────────────────────────────────────────
test_preds = get_scores_and_predict(X_test, len(test_df))

submission = pd.DataFrame({
    "id":         test_df["id"],
    "Prediction": [" ".join(p) for p in test_preds]
})
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")
print(submission.head(10))

Fitting TF-IDF features (word + char ngrams)...
Feature matrix shape: (8000, 53456)
Training Logistic Regression...
Done!

Validation MAP@3: 0.9962
Saved submission.csv
   id Prediction
0   1      A E D
1   2      B E C
2   3      B E D
3   4      E C D
4   5      C A D
5   6      D A C
6   7      E D C
7   8      B E A
8   9      C D E
9  10      B D C


In [4]:
print(submission.head(20))

    id Prediction
0    1      A E D
1    2      B E C
2    3      B E D
3    4      E C D
4    5      C A D
5    6      D A C
6    7      E D C
7    8      B E A
8    9      C D E
9   10      B D C
10  11      A C B
11  12      D E B
12  13      C A E
13  14      C E D
14  15      E D C
15  16      A C E
16  17      E D B
17  18      B E D
18  19      A E B
19  20      D B C


In [ ]:
# # Re-use the few-shot builder and predict function from your earlier run
# few_shot_messages = build_few_shot_examples(train_df, n=3)

# def predict_top3_optimized(prompt, options):
#     opts_text = "\n".join([f"{OPTION_COLS[i]}) {options[i][:300]}" for i in range(5)])
    
#     messages = [
#         {"role": "system", "content": "You are an expert scientist. For each multiple-choice question, analyse the subtle differences between options and rank the top three most likely correct answers."}
#     ]
#     messages.extend(few_shot_messages)
#     messages.append({
#         "role": "user",
#         "content": f"Question: {prompt}\n\nOptions:\n{opts_text}\n\nRank the top 3 correct answers from most likely to least likely.\nReply with ONLY 3 letters separated by spaces. Example: A C B"
#     })
    
#     text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
#     inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
    
#     with torch.no_grad():
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=20,
#             do_sample=False,
#             pad_token_id=tokenizer.eos_token_id
#         )
    
#     new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
#     generated = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    
#     found = []
#     for ch in generated.upper():
#         if ch in OPTION_COLS and ch not in found:
#             found.append(ch)
#         if len(found) == 3:
#             break
    
#     if len(found) < 3:
#         remaining = sorted([c for c in OPTION_COLS if c not in found])
#         found += remaining[:3 - len(found)]
    
#     return found[:3]

# # ------------------------------------------------------------------
# #   VALIDATE ON ONLY 100 SAMPLES (first 100 rows of the split)
# # ------------------------------------------------------------------
# tr, val = train_test_split(train_df, test_size=0.2, random_state=42)
# val100 = val.head(100).reset_index(drop=True)   # <-- use only first 100

# preds, trues = [], []
# for i, row in val100.iterrows():
#     opts = [row[c] for c in OPTION_COLS]
#     pred = predict_top3_optimized(row["prompt"], opts)
#     preds.append(pred)
#     trues.append(row["answer"])
#     if (i+1) % 50 == 0:
#         print(f"  {i+1}/{len(val100)} | MAP@3: {mapk(trues, preds):.4f}")

# final_map = mapk(trues, preds)
# print(f"\nValidation MAP@3 on 100 samples: {final_map:.4f}")



In [ ]:
# from tqdm import tqdm   # ← add this line
# test_preds = []
# for i, row in tqdm(test_df.iterrows(), total=len(test_df)):
#     opts = [row[c] for c in OPTION_COLS]
#     pred = predict_top3_optimized(row["prompt"], opts)
#     test_preds.append(pred)

# submission = pd.DataFrame({
#     "id":         test_df["id"],
#     "Prediction": [" ".join(p) for p in test_preds]
# })
# submission.to_csv("submission.csv", index=False)
# print("Done! submission.csv saved.")

In [ ]:
print(submission)

In [ ]:
# from sklearn.model_selection import train_test_split

# tr, val = train_test_split(train_df, test_size=0.2, random_state=42)
# val = val.reset_index(drop=True)

# print(f"Validating on {len(val)} samples...")

# preds, trues = [], []
# for i, row in val.iterrows():
#     opts = [row[c] for c in OPTION_COLS]
#     pred = predict_top3_optimized(row["prompt"], opts)
#     preds.append(pred)
#     trues.append(row["answer"])
#     if (i+1) % 50 == 0:
#         print(f"  {i+1}/{len(val)} | MAP@3: {mapk(trues, preds):.4f}")

# print(f"\nFinal Validation MAP@3: {mapk(trues, preds):.4f}")

In [ ]:
# test_preds = []

# for i, row in test_df.iterrows():
#     opts = [row[c] for c in OPTION_COLS]
#     pred = predict_top3_optimized(row["prompt"], opts)
#     test_preds.append(pred)
#     if (i+1) % 100 == 0:
#         print(f"{i+1}/{len(test_df)} done")

# submission = pd.DataFrame({
#     "id":         test_df["id"],
#     "Prediction": [" ".join(p) for p in test_preds]
# })
# submission.to_csv("submission.csv", index=False)
# print("Done!")
# print(submission.head(10))

In [ ]:
# print(f"Validation MAP@3: {mapk(trues, preds):.4f}")

In [ ]:
# ##MILESTONE - 1

# import pandas as pd
# import numpy as np
# import string
# from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
# from sklearn.metrics.pairwise import cosine_similarity

# # Load the data
# train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
# test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

# # ============================================
# # Question 1: Frequency distribution of correct answers
# # ============================================
# print("=" * 50)
# print("QUESTION 1: Frequency Distribution of Correct Answers")
# print("=" * 50)

# answer_freq = train_df['answer'].value_counts().sort_index()
# print("Frequency of each answer:")
# print(answer_freq)

# most_frequent = answer_freq.max()
# least_frequent = answer_freq.min()
# sum_most_least = most_frequent + least_frequent

# print(f"\nMost frequent option count: {most_frequent}")
# print(f"Least frequent option count: {least_frequent}")
# print(f"Sum of most and least frequent: {sum_most_least}")

# # ============================================
# # Question 2: Vocabulary size after cleaning prompts
# # ============================================
# print("\n" + "=" * 50)
# print("QUESTION 2: Vocabulary Size After Cleaning")
# print("=" * 50)

# def clean_text(text):
#     # Convert to lowercase
#     text = text.lower()
#     # Remove punctuation
#     text = text.translate(str.maketrans('', '', string.punctuation))
#     return text

# # Clean all prompts
# train_df['cleaned_prompt'] = train_df['prompt'].apply(clean_text)

# # Get all unique words
# all_words = set()
# for prompt in train_df['cleaned_prompt']:
#     words = prompt.split()
#     all_words.update(words)

# vocab_size = len(all_words)
# print(f"Total unique words (vocabulary size): {vocab_size}")

# # ============================================
# # Question 3: Words left in Row ID 1 after removing stop words
# # ============================================
# print("\n" + "=" * 50)
# print("QUESTION 3: Words in Row ID 1 After Removing Stop Words")
# print("=" * 50)

# # Get cleaned prompt for Row ID 1
# row1_prompt = train_df[train_df['id'] == 1]['cleaned_prompt'].values[0]

# # Split into words
# row1_words = row1_prompt.split()

# # Filter out stop words
# row1_filtered = [word for word in row1_words if word not in ENGLISH_STOP_WORDS]

# words_left = len(row1_filtered)
# print(f"Row ID 1 cleaned prompt (first 100 chars): {row1_prompt[:100]}...")
# print(f"Words left after removing stop words: {words_left}")

# # ============================================
# # Question 4: TF-IDF Vectorizer vocabulary size
# # ============================================
# print("\n" + "=" * 50)
# print("QUESTION 4: TF-IDF Vectorizer Vocabulary Size")
# print("=" * 50)

# # Combine prompt and all options into single documents for each row
# combined_texts = []
# for idx, row in train_df.iterrows():
#     # Combine prompt with all options
#     combined = row['prompt'] + ' ' + row['A'] + ' ' + row['B'] + ' ' + row['C'] + ' ' + row['D'] + ' ' + row['E']
#     combined_texts.append(combined)

# # Fit TF-IDF vectorizer
# tfidf_vectorizer = TfidfVectorizer(stop_words='english')
# tfidf_vectorizer.fit(combined_texts)

# feature_columns = len(tfidf_vectorizer.get_feature_names_out())
# print(f"Number of feature columns (vocabulary size): {feature_columns}")

# # ============================================
# # Question 5: Cosine similarity between prompt and option A for Row ID 1
# # ============================================
# print("\n" + "=" * 50)
# print("QUESTION 5: Cosine Similarity for Row ID 1 (Prompt vs Option A)")
# print("=" * 50)

# # Get Row ID 1 data
# row1 = train_df[train_df['id'] == 1].iloc[0]

# # Transform prompt and option A separately
# prompt_vector = tfidf_vectorizer.transform([row1['prompt']])
# option_a_vector = tfidf_vectorizer.transform([row1['A']])

# # Calculate cosine similarity
# similarity_score = cosine_similarity(prompt_vector, option_a_vector)[0][0]

# print(f"Prompt: {row1['prompt'][:100]}...")
# print(f"Option A: {row1['A'][:100]}...")
# print(f"Cosine similarity score: {similarity_score:.4f}")

# # ============================================
# # Question 6: Percentage where highest similarity matches correct answer
# # ============================================
# print("\n" + "=" * 50)
# print("QUESTION 6: Percentage of Highest Similarity Matching Correct Answer")
# print("=" * 50)

# correct_matches = 0
# total_rows = len(train_df)

# for idx, row in train_df.iterrows():
#     # Vectorize prompt
#     prompt_vec = tfidf_vectorizer.transform([row['prompt']])
    
#     # Vectorize each option and calculate similarity
    
#     similarities = {}
#     for option in ['A', 'B', 'C', 'D', 'E']:
#         option_vec = tfidf_vectorizer.transform([row[option]])
#         sim = cosine_similarity(prompt_vec, option_vec)[0][0]
#         similarities[option] = sim
    
#     # Find option with highest similarity
#     highest_sim_option = max(similarities, key=similarities.get)
    
#     # Check if matches correct answer
#     if highest_sim_option == row['answer']:
#         correct_matches += 1

# percentage = (correct_matches / total_rows) * 100
# print(f"Rows where highest similarity matches correct answer: {correct_matches}/{total_rows}")
# print(f"Percentage: {percentage:.2f}%")

# # ============================================
# # Question 7: MAP@3 score for prediction C A B when answer is C
# # ============================================
# print("\n" + "=" * 50)
# print("QUESTION 7: MAP@3 for prediction C A B (answer is C)")
# print("=" * 50)

# def calculate_map_at_3(ground_truth, predictions):
#     """
#     Calculate MAP@3 for a single question
#     predictions: list of 3 predicted answers in order
#     """
#     for i, pred in enumerate(predictions):
#         if pred == ground_truth:
#             return 1.0 / (i + 1)  # 1/k where k is the position (1-indexed)
#     return 0.0  # Not in top 3

# # Example: answer is C, prediction is C A B
# map_score_q7 = calculate_map_at_3('C', ['C', 'A', 'B'])
# print(f"Ground truth: C, Prediction: C A B")
# print(f"MAP@3 score: {map_score_q7}")

# # ============================================
# # Question 8: MAP@3 score for prediction D B E when answer is B
# # ============================================
# print("\n" + "=" * 50)
# print("QUESTION 8: MAP@3 for prediction D B E (answer is B)")
# print("=" * 50)

# map_score_q8 = calculate_map_at_3('B', ['D', 'B', 'E'])
# print(f"Ground truth: B, Prediction: D B E")
# print(f"MAP@3 score: {map_score_q8}")

# # ============================================
# # Question 9: Majority Class Baseline MAP@3
# # ============================================
# print("\n" + "=" * 50)
# print("QUESTION 9: Majority Class Baseline MAP@3")
# print("=" * 50)

# # Get frequency of answers
# answer_counts = train_df['answer'].value_counts()
# print("Answer frequencies:")
# print(answer_counts)

# # Get top 3 most frequent answers
# top3_answers = answer_counts.head(3).index.tolist()
# print(f"Top 3 most frequent answers: {top3_answers}")

# # Calculate MAP@3 for majority baseline
# majority_scores = []
# for idx, row in train_df.iterrows():
#     ground_truth = row['answer']
#     predictions = top3_answers  # Always predict the same top 3
#     score = calculate_map_at_3(ground_truth, predictions)
#     majority_scores.append(score)

# overall_majority_map = np.mean(majority_scores)
# print(f"Overall MAP@3 for Majority Class Baseline: {overall_majority_map:.4f}")

# # ============================================
# # Question 10: TF-IDF Pipeline MAP@3
# # ============================================
# print("\n" + "=" * 50)
# print("QUESTION 10: TF-IDF Pipeline MAP@3")
# print("=" * 50)

# tfidf_scores = []

# for idx, row in train_df.iterrows():
#     # Vectorize prompt
#     prompt_vec = tfidf_vectorizer.transform([row['prompt']])
    
#     # Calculate similarity for each option
#     similarities = {}
#     for option in ['A', 'B', 'C', 'D', 'E']:
#         option_vec = tfidf_vectorizer.transform([row[option]])
#         sim = cosine_similarity(prompt_vec, option_vec)[0][0]
#         similarities[option] = sim
    
#     # Sort options by similarity (highest to lowest)
#     sorted_options = sorted(similarities.items(), key=lambda x: x[1], reverse=True)
    
#     # Get top 3 predictions
#     top3_predictions = [option for option, sim in sorted_options[:3]]
    
#     # Calculate MAP@3 for this row
#     ground_truth = row['answer']
#     score = calculate_map_at_3(ground_truth, top3_predictions)
#     tfidf_scores.append(score)

# overall_tfidf_map = np.mean(tfidf_scores)
# print(f"Overall MAP@3 for TF-IDF Pipeline: {overall_tfidf_map:.4f}")

# # ============================================
# # Create sample submission file
# # ============================================
# print("\n" + "=" * 50)
# print("Creating Sample Submission File")
# print("=" * 50)

# # Create predictions for test set using TF-IDF approach
# submission_predictions = []

# for idx, row in test_df.iterrows():
#     prompt_vec = tfidf_vectorizer.transform([row['prompt']])
    
#     similarities = {}
#     for option in ['A', 'B', 'C', 'D', 'E']:
#         option_vec = tfidf_vectorizer.transform([row[option]])
#         sim = cosine_similarity(prompt_vec, option_vec)[0][0]
#         similarities[option] = sim
    
#     sorted_options = sorted(similarities.items(), key=lambda x: x[1], reverse=True)
#     top3_predictions = [option for option, sim in sorted_options[:3]]
    
#     submission_predictions.append({
#         'ID': row['id'],
#         'Prediction': ' '.join(top3_predictions)
#     })

# # Create submission DataFrame
# submission_df = pd.DataFrame(submission_predictions)

# # Save to CSV
# submission_df.to_csv('sample_submission.csv', index=False)

# print(f"Sample submission file created with {len(submission_df)} predictions")
# print("\nFirst few predictions:")
# print(submission_df.head())